# MOD-04 · NB-05 — Hyperparameter Tuning & Cross-Validation Strategies
### Health Analytics with Python · Module 04: Machine Learning for Clinical Prediction
---
**Learning objectives**
- Compare grid search, random search, and Bayesian optimisation (Optuna)
- Implement nested cross-validation for unbiased performance estimation
- Use Halving GridSearch for efficient large-scale tuning
- Build learning curves to diagnose underfitting and overfitting
- Apply regularisation paths (Ridge, LASSO, ElasticNet logistic regression)

**Estimated time:** 2 hours | **Level:** Advanced | **Libraries:** `sklearn`, `optuna`, `matplotlib`


## 1. Setup

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                      RandomizedSearchCV, cross_val_score, learning_curve)
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from scipy.stats import uniform, randint, loguniform
import warnings; warnings.filterwarnings("ignore")
import os; os.makedirs("/tmp/mod04",exist_ok=True)
plt.rcParams.update({"figure.dpi":120,"figure.facecolor":"white",
                     "axes.spines.top":False,"axes.spines.right":False})

# Rebuild dataset (same seed)
np.random.seed(42); N=3000
def sig(x): return 1/(1+np.exp(-x))
b=np.random.normal(size=(N,4)); np.random.seed(99)
age=np.random.normal(62,15,N).clip(18,92).astype(int)
sex=np.random.choice(["M","F"],N,p=[0.48,0.52])
payer=np.random.choice(["Medicare","Medicaid","Private","Self-pay","Other"],N,p=[0.40,0.22,0.28,0.07,0.03])
admit_type=np.random.choice(["Emergency","Elective","Urgent"],N,p=[0.52,0.30,0.18])
los_days=np.random.gamma(2.5,1.8,N).clip(1,30).astype(int)
diabetes=np.random.binomial(1,sig(0.6*b[:,0]-0.5),N)
hypertension=np.random.binomial(1,sig(0.7*b[:,0]-0.2),N)
hf=np.random.binomial(1,sig(0.4*b[:,1]+0.5*hypertension-1.5),N)
copd=np.random.binomial(1,sig(0.5*b[:,2]-1.0),N)
ckd=np.random.binomial(1,sig(0.5*b[:,0]+0.6*diabetes-1.2),N)
obesity=np.random.binomial(1,sig(0.5*b[:,0]-0.8),N)
depression=np.random.binomial(1,sig(0.3*b[:,3]-1.4),N)
n_comorb=diabetes+hypertension+hf+copd+ckd
np.random.seed(21)
glucose=np.random.normal(105+15*diabetes,28,N).clip(50,400).round(1)
creatinine=np.random.gamma(1.6+0.8*hf,0.6,N).clip(0.4,10).round(2)
hba1c=np.random.normal(6.5+0.8*diabetes,1.4,N).clip(4,14).round(1)
sbp=np.random.normal(128+12*hypertension,18,N).clip(80,200).round(0)
bmi=np.random.normal(28+4*obesity,6,N).clip(15,55).round(1)
logit=(-3.2+0.6*hf+0.5*diabetes+0.5*ckd+0.3*copd+0.02*(age-62)/15
       +0.3*(admit_type=="Emergency").astype(int)+0.25*(payer=="Medicaid").astype(int)
       +0.2*(los_days>7).astype(int)+0.5*hf*diabetes+np.random.normal(0,0.25,N))
readmitted=np.random.binomial(1,sig(logit),N)
df=pd.DataFrame({
    "age":age,"sex":sex,"payer":payer,"admit_type":admit_type,"los_days":los_days,
    "diabetes":diabetes,"hypertension":hypertension,"hf":hf,"copd":copd,"ckd":ckd,
    "obesity":obesity,"depression":depression,"n_comorb":n_comorb,
    "glucose":glucose,"creatinine":creatinine,"hba1c":hba1c,"sbp":sbp,"bmi":bmi,"readmitted":readmitted,
})
df["los_gt7"]=(df.los_days>7).astype(int)
NUMERIC=["age","los_days","n_comorb","glucose","creatinine","hba1c","sbp","bmi"]
BINARY=["diabetes","hypertension","hf","copd","ckd","obesity","depression","los_gt7"]
CATEGORIC=["sex","payer","admit_type"]
FEATURES=NUMERIC+BINARY+CATEGORIC; TARGET="readmitted"
pre=ColumnTransformer([
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())]),NUMERIC),
    ("bin",SimpleImputer(strategy="most_frequent"),BINARY),
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),
                     ("ohe",OneHotEncoder(handle_unknown="ignore",sparse_output=False))]),CATEGORIC),
])
X=df[FEATURES]; y=df[TARGET]
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=0.20,stratify=y,random_state=42)
pre.fit(X_tr); Xtr=pre.transform(X_tr); Xte=pre.transform(X_te)
print(f"Ready: Train {Xtr.shape}, Test {Xte.shape}")


## 2. Grid Search vs Random Search vs Halving

In [ ]:
import time

# Grid search
param_grid={"n_estimators":[100,200],"max_depth":[4,6,8],"learning_rate":[0.05,0.10],"subsample":[0.7,0.9]}
t0=time.time()
gs=GridSearchCV(GradientBoostingClassifier(random_state=42),param_grid,
                cv=StratifiedKFold(3,shuffle=True,random_state=42),scoring="roc_auc",n_jobs=-1)
gs.fit(Xtr,y_tr); t_grid=time.time()-t0
print(f"Grid search    : {len(gs.cv_results_['params']):3d} configs | best AUC={gs.best_score_:.4f} | time={t_grid:.1f}s")

# Random search
from scipy.stats import randint, loguniform, uniform
param_dist={"n_estimators":randint(50,400),"max_depth":randint(3,10),
            "learning_rate":loguniform(0.01,0.3),"subsample":uniform(0.5,0.5),
            "min_samples_leaf":randint(5,30)}
t0=time.time()
rs=RandomizedSearchCV(GradientBoostingClassifier(random_state=42),param_dist,n_iter=30,
                      cv=StratifiedKFold(3,shuffle=True,random_state=42),scoring="roc_auc",
                      random_state=42,n_jobs=-1)
rs.fit(Xtr,y_tr); t_rand=time.time()-t0
print(f"Random search  :  30 configs | best AUC={rs.best_score_:.4f} | time={t_rand:.1f}s")

# Successive halving
t0=time.time()
hs=HalvingGridSearchCV(GradientBoostingClassifier(random_state=42),param_grid,
                       cv=StratifiedKFold(3,shuffle=True,random_state=42),
                       scoring="roc_auc",factor=3,n_jobs=-1,random_state=42)
hs.fit(Xtr,y_tr); t_halv=time.time()-t0
print(f"Halving grid   : auto reduce | best AUC={hs.best_score_:.4f} | time={t_halv:.1f}s")
print(f"
Halving speedup vs grid search: {t_grid/max(t_halv,0.01):.1f}×")


## 3. Nested Cross-Validation — Unbiased Evaluation

In [ ]:
# Nested CV: inner loop tunes hyperparams, outer loop evaluates generalisation
# This avoids the optimistic bias of tuning and evaluating on the same data
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv  = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

param_grid_small = {"n_estimators":[100,200],"max_depth":[4,6],"learning_rate":[0.05,0.10]}
nested_model = GradientBoostingClassifier(random_state=42)

nested_auc = []
for fold,(tr_idx,te_idx) in enumerate(outer_cv.split(Xtr,y_tr)):
    X_in,X_out = Xtr[tr_idx],Xtr[te_idx]
    y_in,y_out = y_tr.iloc[tr_idx],y_tr.iloc[te_idx]
    gs_inner = GridSearchCV(nested_model,param_grid_small,cv=inner_cv,scoring="roc_auc",n_jobs=-1)
    gs_inner.fit(X_in,y_in)
    prob_out = gs_inner.best_estimator_.predict_proba(X_out)[:,1]
    auc_out  = roc_auc_score(y_out,prob_out)
    nested_auc.append(auc_out)
    print(f"  Outer fold {fold+1}: best_params={gs_inner.best_params_} | AUC={auc_out:.4f}")

print(f"
Nested CV AUC: {np.mean(nested_auc):.4f} ± {np.std(nested_auc):.4f}")
print(f"Non-nested (grid search, same data): {gs.best_score_:.4f}")
print(f"Optimism bias: {gs.best_score_-np.mean(nested_auc):+.4f}")


## 4. Learning Curves

In [ ]:
best_gb = gs.best_estimator_
train_sizes, train_scores, val_scores = learning_curve(
    best_gb, Xtr, y_tr,
    train_sizes=np.linspace(0.10,1.0,10),
    cv=StratifiedKFold(3,shuffle=True,random_state=42),
    scoring="roc_auc", n_jobs=-1
)
fig,ax=plt.subplots(figsize=(10,5))
ax.plot(train_sizes,train_scores.mean(axis=1),"-o",color="#D65F5F",lw=2,ms=6,label="Train AUC")
ax.fill_between(train_sizes,
                train_scores.mean(axis=1)-train_scores.std(axis=1),
                train_scores.mean(axis=1)+train_scores.std(axis=1),alpha=0.15,color="#D65F5F")
ax.plot(train_sizes,val_scores.mean(axis=1),"-o",color="#4878CF",lw=2,ms=6,label="CV Val AUC")
ax.fill_between(train_sizes,
                val_scores.mean(axis=1)-val_scores.std(axis=1),
                val_scores.mean(axis=1)+val_scores.std(axis=1),alpha=0.15,color="#4878CF")
gap=train_scores.mean(axis=1)[-1]-val_scores.mean(axis=1)[-1]
ax.text(0.05,0.15,f"Train-val gap at N={train_sizes[-1]:.0f}: {gap:.3f}",
        transform=ax.transAxes,fontsize=10,
        bbox=dict(facecolor="white",alpha=0.8,edgecolor="none"))
ax.set_xlabel("Training set size"); ax.set_ylabel("AUC-ROC")
ax.set_title("Learning curves — diagnosis of over/underfitting")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/tmp/mod04/learning_curves.png",bbox_inches="tight"); plt.show()
print("Diagnosing learning curves:")
print("  Large gap (train >> val) → overfitting → increase regularisation or reduce complexity")
print("  Both curves low          → underfitting → add features, reduce regularisation, or use a more complex model")
print("  Curves converge          → more data won't help; focus on feature engineering")


## 5. Regularisation Paths for Logistic Regression

In [ ]:
Cs = np.logspace(-4,4,30)
aucs_train, aucs_val = [], []
for C in Cs:
    lr=LogisticRegression(C=C,max_iter=500,class_weight="balanced",random_state=42)
    lr.fit(Xtr,y_tr)
    aucs_train.append(roc_auc_score(y_tr,lr.predict_proba(Xtr)[:,1]))
    aucs_val.append(roc_auc_score(y_te,lr.predict_proba(Xte)[:,1]))

best_C = Cs[np.argmax(aucs_val)]
fig,ax=plt.subplots(figsize=(10,5))
ax.semilogx(Cs,aucs_train,"-",color="#D65F5F",lw=2,label="Train AUC")
ax.semilogx(Cs,aucs_val,  "-",color="#4878CF",lw=2,label="Test AUC")
ax.axvline(best_C,color="green",ls="--",lw=1.5,label=f"Best C={best_C:.4f}")
ax.axvspan(Cs[0],1e-1,alpha=0.05,color="blue",label="High regularisation (underfitting)")
ax.axvspan(1e2,Cs[-1],alpha=0.05,color="red", label="Low regularisation (overfitting)")
ax.set_xlabel("C (inverse regularisation strength)"); ax.set_ylabel("AUC-ROC")
ax.set_title("Regularisation path — Logistic Regression")
ax.legend(fontsize=8); ax.set_ylim(0.5,0.90)
plt.tight_layout(); plt.savefig("/tmp/mod04/reg_path.png",bbox_inches="tight"); plt.show()
print(f"Best C: {best_C:.4f} | Best test AUC: {max(aucs_val):.4f}")


## 6. Knowledge Check
1. Nested CV AUC = 0.76 but standard CV AUC = 0.79. What does the 0.03 gap represent?
2. Your learning curves show train AUC=0.91 and val AUC=0.74 at N=2000. What should you do?
3. A very small C (say 0.001) in logistic regression leads to what kind of bias-variance problem?
4. HalvingGridSearch allocated more budget to promising configurations. Explain this mechanism in two sentences.
5. Re-run nested CV using Random Forest instead of GBM. Does the optimism bias increase or decrease?


In [ ]:
# Exercise 5 — nested CV with RF
from sklearn.ensemble import RandomForestClassifier
rf_nested_auc=[]
for fold,(tr_idx,te_idx) in enumerate(outer_cv.split(Xtr,y_tr)):
    X_in,X_out=Xtr[tr_idx],Xtr[te_idx]
    y_in,y_out=y_tr.iloc[tr_idx],y_tr.iloc[te_idx]
    pg_rf={"n_estimators":[100,200],"max_depth":[6,10],"max_features":["sqrt","log2"]}
    gs_rf=GridSearchCV(RandomForestClassifier(class_weight="balanced",random_state=42),
                       pg_rf,cv=inner_cv,scoring="roc_auc",n_jobs=-1)
    gs_rf.fit(X_in,y_in)
    rf_nested_auc.append(roc_auc_score(y_out,gs_rf.best_estimator_.predict_proba(X_out)[:,1]))
print(f"RF Nested CV : {np.mean(rf_nested_auc):.4f} ± {np.std(rf_nested_auc):.4f}")
print(f"GB Nested CV : {np.mean(nested_auc):.4f} ± {np.std(nested_auc):.4f}")


---
**Next → NB-06: SHAP Explainability for Clinical Models**